# 🎬 Human Activity Recognition — VideoMAE (Pretrained)
**Computer Vision Course Project**

Adam El Kaissi - 2101431 || Awais Ahmed - 2281583

---

This notebook uses **VideoMAE**, a state-of-the-art pretrained video understanding model (trained on Kinetics-400, 400 action classes). No dataset download or training required.

### ⚡ Steps
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells top to bottom
3. In Cell 4, upload your own video files to test — or use the sample download cell

Total runtime: **~5 minutes** (mostly model download)

## Cell 1 — Check GPU & install dependencies

In [ ]:
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU: {result.stdout.strip()}')
else:
    print('❌ No GPU — go to Runtime → Change runtime type → T4 GPU')
    raise SystemExit

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'accelerate', 'opencv-python-headless',
                'scikit-learn', 'seaborn', 'matplotlib'], check=True)
print('✅ Dependencies installed.')

## Cell 2 — Load pretrained VideoMAE model

In [ ]:
import torch
from transformers import VideoMAEForVideoClassification, AutoProcessor

MODEL_NAME = 'MCG-NJU/videomae-base-finetuned-kinetics'
NUM_FRAMES = 16
OUTPUT_DIR = '/content/outputs'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Loading {MODEL_NAME} ...')

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model     = VideoMAEForVideoClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

# Class labels
id2label = model.config.id2label
label2id = model.config.label2id
print(f'✅ Model loaded — {len(id2label)} action classes')
print(f'Example classes: {", ".join(list(id2label.values())[:8])}')

## Cell 3 — Download sample test videos
These are short royalty-free clips from Wikimedia. Skip this cell and upload your own videos in Cell 4 if you prefer.

In [ ]:
import subprocess, os

os.makedirs('/content/test_videos', exist_ok=True)

# Royalty-free sample clips from Wikimedia Commons
sample_videos = [
    ('https://upload.wikimedia.org/wikipedia/commons/transcoded/b/b3/Big_Buck_Bunny_Trailer_400p.ogv/Big_Buck_Bunny_Trailer_400p.ogv.360p.webm',
     'sample_1.webm'),
    ('https://upload.wikimedia.org/wikipedia/commons/transcoded/0/02/Four_Five_Seconds_Music_Video_-_Behind_the_Scenes_footage.webm/Four_Five_Seconds_Music_Video_-_Behind_the_Scenes_footage.webm.360p.webm',
     'sample_2.webm'),
]

# Better: download specific action clips using yt-dlp from YouTube
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp'], check=True)

# Short YouTube clips of clear human actions (public domain / creative commons)
clips = [
    ('https://www.youtube.com/watch?v=ixjWEAu-SaQ', 'juggling.mp4'),
    ('https://www.youtube.com/watch?v=2pLT-olgUJs', 'jumping_jacks.mp4'),
    ('https://www.youtube.com/watch?v=IODxDxX7oi4', 'push_ups.mp4'),
]

print('Downloading sample action videos...')
for url, fname in clips:
    out_path = f'/content/test_videos/{fname}'
    if os.path.exists(out_path):
        print(f'  Already exists: {fname}')
        continue
    result = subprocess.run(
        ['yt-dlp', '-q', '--no-warnings', '-f', 'mp4/best[height<=480]',
         '--output', out_path, url],
        capture_output=True, text=True
    )
    if result.returncode == 0 and os.path.exists(out_path):
        print(f'  ✅ {fname}')
    else:
        print(f'  ⚠️  Could not download {fname} — upload your own video instead')

videos = [f'/content/test_videos/{f}' for f in os.listdir('/content/test_videos')
          if f.endswith(('.mp4', '.avi', '.webm', '.mov'))]
print(f'\n{len(videos)} test videos ready: {[os.path.basename(v) for v in videos]}')

## Cell 4 — Upload your own videos (optional)
Run this to upload videos from your computer instead of (or in addition to) the sample ones above.

In [ ]:
from google.colab import files
import shutil

print('Select video files to upload (.mp4, .avi, .mov)...')
uploaded = files.upload()

os.makedirs('/content/test_videos', exist_ok=True)
for fname, data in uploaded.items():
    dest = f'/content/test_videos/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'✅ Saved: {dest}')

videos = sorted([f'/content/test_videos/{f}' for f in os.listdir('/content/test_videos')
                 if f.endswith(('.mp4', '.avi', '.webm', '.mov'))])
print(f'\nTotal videos ready: {len(videos)}')

## Cell 5 — Frame extraction & inference helper

In [ ]:
import cv2
import numpy as np
import torch

def extract_frames(video_path, num_frames=16):
    """Uniformly sample num_frames frames from a video. Returns list of RGB numpy arrays."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1:
        cap.release()
        return None
    indices = np.linspace(0, max(total - 1, 0), num=num_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        elif frames:
            frames.append(frames[-1].copy())
    cap.release()
    return frames if len(frames) == num_frames else None

@torch.no_grad()
def predict(video_path, top_k=5):
    """Run VideoMAE on a video file. Returns top-k (label, confidence) pairs."""
    frames = extract_frames(video_path, NUM_FRAMES)
    if frames is None:
        return None

    inputs = processor(images=frames, return_tensors='pt').to(device)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).squeeze()

    top = probs.topk(top_k)
    return [(id2label[idx.item()], round(prob.item() * 100, 1))
            for prob, idx in zip(top.values, top.indices)]

print('✅ Inference functions ready.')

## Cell 6 — Run predictions on all test videos

In [ ]:
import os

videos = sorted([f'/content/test_videos/{f}' for f in os.listdir('/content/test_videos')
                 if f.endswith(('.mp4', '.avi', '.webm', '.mov'))])

if not videos:
    print('⚠️  No videos found in /content/test_videos — run Cell 3 or Cell 4 first.')
else:
    all_results = {}
    for video_path in videos:
        name = os.path.basename(video_path)
        print(f'\n🎬 {name}')
        print('─' * 40)
        preds = predict(video_path, top_k=5)
        if preds is None:
            print('  ⚠️  Could not read video')
            continue
        all_results[name] = preds
        for rank, (label, conf) in enumerate(preds, 1):
            bar = '█' * int(conf / 5)
            marker = '👉 ' if rank == 1 else f'  {rank}.'
            print(f'{marker} {label:<35} {conf:5.1f}%  {bar}')

    print(f'\n✅ Done — {len(all_results)} videos processed.')

## Cell 7 — Visualise frame samples from each video

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

videos = sorted([f'/content/test_videos/{f}' for f in os.listdir('/content/test_videos')
                 if f.endswith(('.mp4', '.avi', '.webm', '.mov'))])

for video_path in videos:
    name = os.path.basename(video_path)
    frames = extract_frames(video_path, num_frames=8)
    if frames is None:
        continue

    preds = all_results.get(name, [])
    top_label = preds[0][0] if preds else 'unknown'
    top_conf  = preds[0][1] if preds else 0

    fig, axes = plt.subplots(1, 8, figsize=(18, 2.5))
    fig.suptitle(f'{name}   →   Predicted: "{top_label}"  ({top_conf}% confidence)',
                 fontsize=11, fontweight='bold', y=1.02)
    for ax, frame in zip(axes, frames):
        ax.imshow(frame)
        ax.axis('off')
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, f'frames_{os.path.splitext(name)[0]}.png')
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {save_path}')

## Cell 8 — Confidence bar chart (report figure)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for video_name, preds in all_results.items():
    labels = [p[0].replace('_', ' ') for p in preds]
    confs  = [p[1] for p in preds]
    colors = ['#1a7abf' if i == 0 else '#aec6d8' for i in range(len(preds))]

    fig, ax = plt.subplots(figsize=(8, 3.5))
    bars = ax.barh(labels[::-1], confs[::-1], color=colors[::-1], edgecolor='white')
    ax.set_xlabel('Confidence (%)', fontsize=11)
    ax.set_title(f'Top-5 Predictions — {video_name}', fontsize=12, fontweight='bold')
    ax.set_xlim(0, max(confs) * 1.2)
    for bar, val in zip(bars, confs[::-1]):
        ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=10)
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, f'predictions_{os.path.splitext(video_name)[0]}.png')
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f'Saved → {save_path}')

## Cell 9 — Evaluate on multiple videos with known labels
If you have videos where you know the correct action, add them here to get accuracy metrics.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Fill this in with your test videos and their true action labels
# Format: (video_path, true_label)
# true_label must match a Kinetics-400 class name exactly
# Run Cell 2 and check id2label.values() to see all valid class names
labeled_videos = [
    # ('/content/test_videos/juggling.mp4',      'juggling balls'),
    # ('/content/test_videos/jumping_jacks.mp4', 'jumping jacks'),
    # ('/content/test_videos/push_ups.mp4',      'push up'),
]

if not labeled_videos:
    print('ℹ️  No labeled videos defined — add your videos and true labels above to get metrics.')
    print('    This cell is optional; used for the evaluation section of your report.')
else:
    y_true, y_pred = [], []
    for video_path, true_label in labeled_videos:
        preds = predict(video_path, top_k=1)
        if preds is None:
            continue
        predicted_label = preds[0][0]
        y_true.append(true_label)
        y_pred.append(predicted_label)
        correct = '✅' if predicted_label == true_label else '❌'
        print(f'{correct} True: {true_label:<30} Predicted: {predicted_label}')

    top1 = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true) * 100
    print(f'\nTop-1 Accuracy: {top1:.1f}%  ({sum(t==p for t,p in zip(y_true,y_pred))}/{len(y_true)} correct)')
    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, zero_division=0))

## Cell 10 — Save results summary & download outputs

In [ ]:
import json, shutil
from google.colab import files

# Save predictions to JSON
results_path = os.path.join(OUTPUT_DIR, 'predictions.json')
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Predictions saved → {results_path}')

# Save summary text file
summary_path = os.path.join(OUTPUT_DIR, 'results_summary.txt')
with open(summary_path, 'w') as f:
    f.write('Human Activity Recognition — VideoMAE Results\n')
    f.write('=' * 50 + '\n\n')
    f.write(f'Model: {MODEL_NAME}\n')
    f.write(f'Frames per clip: {NUM_FRAMES}\n')
    f.write(f'Videos tested: {len(all_results)}\n\n')
    for video_name, preds in all_results.items():
        f.write(f'{video_name}:\n')
        for rank, (label, conf) in enumerate(preds, 1):
            f.write(f'  {rank}. {label} ({conf}%)\n')
        f.write('\n')
print(f'Summary saved → {summary_path}')

# Zip and download everything
shutil.make_archive('/content/har_outputs', 'zip', OUTPUT_DIR)
print('\n⬇️  Downloading outputs zip...')
files.download('/content/har_outputs.zip')
print('Done!')